# Hybrid Quantization: MXFP4 + IQ4_NL + IQ4_XS

**Strategy:**
- `mxfp4` tensors (expert weights) → `copy` (keep as-is)
- 2880-column tensors (not divisible by 256) → `iq4_nl` (32-block, no fallback)
- everything else → `IQ4_XS` + imatrix (best quality for attention/embed)

**Steps:**
1. Discover mxfp4 tensors and 2880-column fallback tensors
2. Export calibration data from SFT training dataset
3. Generate imatrix
4. Run custom quantization

In [ ]:
import os, re, subprocess

MXFP4_GGUF    = "gpt-oss-20b.MXFP4.gguf"
LLAMA_DIR     = "llama.cpp"
QUANTIZER     = f"{LLAMA_DIR}/llama-quantize"
IMATRIX_BIN   = f"{LLAMA_DIR}/llama-imatrix"
CALIB_FILE    = "calibration.txt"
IMATRIX_FILE  = "gpt-oss-20b-imatrix.dat"
OUTPUT_GGUF   = "gpt-oss-20b-sft-IQ4_XS-hybrid.gguf"
LD_PATH       = f"{LLAMA_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ENV           = {**os.environ, "LD_LIBRARY_PATH": LD_PATH}

for f in [MXFP4_GGUF, QUANTIZER, IMATRIX_BIN]:
    status = "✅" if os.path.exists(f) else "❌"
    print(f"{status} {f}")

## Step 1 — Discover MXFP4 tensors and 2880-column fallback tensors

Run this in the terminal first, then come back to parse the log:

```bash
LD_LIBRARY_PATH=llama.cpp:$LD_LIBRARY_PATH llama.cpp/llama-quantize \
  --allow-requantize \
  gpt-oss-20b.MXFP4.gguf \
  /tmp/dry.gguf \
  IQ4_XS 2>&1 | tee quantize_dry_run.log

rm /tmp/dry.gguf
```

In [ ]:
LOG_FILE = "quantize_dry_run.log"

if not os.path.exists(LOG_FILE):
    raise FileNotFoundError(
        f"{LOG_FILE} not found.\n"
        "Run the terminal command in the markdown cell above first."
    )

with open(LOG_FILE) as f:
    output = f.read()

print(f"Log file: {len(output.splitlines())} lines")

# Parse mxfp4 tensors
mxfp4_tensors = set()
for line in output.splitlines():
    if "mxfp4" in line and "]" in line:
        m = re.search(r"\]\s+(\S+)\s+-", line)
        if m:
            generic = re.sub(r"blk\.\d+\.", "blk.*.", m.group(1))
            mxfp4_tensors.add(generic)

# Parse fallback tensors (ncols not divisible by 256)
fallback_tensors = set()
for line in output.splitlines():
    if "not divisible" in line or "falling back" in line:
        m = re.search(r"warning:\s+(\S+)\s+-", line)
        if m:
            generic = re.sub(r"blk\.\d+\.", "blk.*.", m.group(1))
            fallback_tensors.add(generic)

fallback_tensors -= mxfp4_tensors

print(f"\nMXFP4 tensors (will COPY): {len(mxfp4_tensors)}")
for t in sorted(mxfp4_tensors): print(f"  {t}")

print(f"\nFallback tensors (will use IQ4_NL): {len(fallback_tensors)}")
for t in sorted(fallback_tensors): print(f"  {t}")

## Step 2 — Calibration data from SFT training dataset

In [ ]:
# Tries local SFT dataset first, falls back to wikitext.

CALIB_SAMPLES = 512

if os.path.exists(CALIB_FILE):
    print(f"✅ {CALIB_FILE} already exists, skipping.")
else:
    # --- Option A: local SFT dataset ---
    sft_written = False
    for ds_path in ["sft_dataset", "datasets/sft", "data/sft"]:
        if os.path.isdir(ds_path):
            from datasets import load_from_disk
            print(f"Loading SFT dataset from {ds_path} ...")
            ds = load_from_disk(ds_path)
            # handle DatasetDict (train split)
            if hasattr(ds, "keys"):
                ds = ds["train"]
            samples = min(CALIB_SAMPLES, len(ds))
            text_col = next((c for c in ["text", "content", "prompt"] if c in ds.column_names), None)
            if text_col is None:
                print(f"  Columns: {ds.column_names} — set text_col manually below")
            else:
                with open(CALIB_FILE, "w") as f:
                    for row in ds.select(range(samples)):
                        text = row[text_col].strip()
                        if text:
                            f.write(text + "\n")
                print(f"✅ Wrote {samples} samples from SFT dataset → {CALIB_FILE}")
                sft_written = True
            break

    # --- Option B: HuggingFace SFT datasets used in training ---
    if not sft_written and not os.path.exists(CALIB_FILE):
        print("No local SFT dataset found. Loading open-thoughts from HuggingFace ...")
        from datasets import load_dataset
        ds = load_dataset("open-thoughts/OpenThoughts-114k", split="train")
        with open(CALIB_FILE, "w") as f:
            for row in ds.select(range(CALIB_SAMPLES)):
                # combine system + user + assistant for richer calibration
                parts = []
                for msg in row.get("conversations", []):
                    parts.append(msg.get("value", "").strip())
                text = " ".join(parts).strip()
                if text:
                    f.write(text + "\n")
        print(f"✅ Wrote {CALIB_SAMPLES} samples → {CALIB_FILE}")

# Verify
with open(CALIB_FILE) as f:
    lines = f.readlines()
print(f"Calibration file: {len(lines)} lines, {os.path.getsize(CALIB_FILE)/1e6:.1f} MB")

## Step 3 — Generate imatrix (~20 min on A100)

In [ ]:
if os.path.exists(IMATRIX_FILE):
    print(f"✅ {IMATRIX_FILE} already exists, skipping.")
else:
    cmd = [
        IMATRIX_BIN,
        "-m", MXFP4_GGUF,
        "-f", CALIB_FILE,
        "-o", IMATRIX_FILE,
        "-ngl", "99",
        "--chunks", "128",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=ENV)
    print(f"\n✅ {IMATRIX_FILE} ready ({os.path.getsize(IMATRIX_FILE)/1e6:.1f} MB)")

## Step 4 — Hybrid quantization: MXFP4 copy + IQ4_NL + IQ4_XS

In [ ]:
# Build --tensor-type args dynamically from Step 1 discoveries

tensor_type_args = []

# MXFP4 expert weights → copy (keep as-is)
for t in sorted(mxfp4_tensors):
    tensor_type_args += ["--tensor-type", f"{t}=copy"]

# 2880-column fallback tensors → iq4_nl (32-block, no fallback)
for t in sorted(fallback_tensors):
    tensor_type_args += ["--tensor-type", f"{t}=iq4_nl"]

print("Tensor type overrides:")
for i in range(0, len(tensor_type_args), 2):
    print(f"  {tensor_type_args[i+1]}")

cmd = [
    QUANTIZER,
    "--allow-requantize",
    "--imatrix", IMATRIX_FILE,
] + tensor_type_args + [
    MXFP4_GGUF,
    OUTPUT_GGUF,
    "IQ4_XS",
]

print(f"\nOutput: {OUTPUT_GGUF}")
print("Running quantization ...\n")
subprocess.run(cmd, check=True, env=ENV)

size_gb = os.path.getsize(OUTPUT_GGUF) / 1e9
print(f"\n✅ Done: {OUTPUT_GGUF} — {size_gb:.1f} GB")

In [ ]:
# Quick inference test
cmd = [
    f"{LLAMA_DIR}/llama-cli",
    "--model", OUTPUT_GGUF,
    "--n-gpu-layers", "99",
    "-p", "Explain step by step why the sky is blue.",
    "-n", "256",
    "--log-disable",
]
result = subprocess.run(cmd, capture_output=True, text=True, env=ENV)
print(result.stdout[-3000:] if result.stdout else result.stderr[-1000:])